# 🔧 Divvy Bike Usage Prediction - Feature Engineering

**Project:** Prédiction de l'utilisation des vélos Divvy à Chicago  
**Author:** Noé  
**Date:** 2026-01-13  
**Objective:** Créer des features simples et explicables pour la modélisation

---

## 🎯 Objectif du notebook

**Prédire le nombre de trajets PAR HEURE** pour savoir combien de vélos sont nécessaires

### Features à créer:
- **Temporelles**: hour, day_of_week, month, is_weekend, season
- **Météo**: temperature, precipitation, wind_speed (agrégées au jour)
- **Calendrier**: is_holiday

### Approche simple (MVP):
- ✅ Agrégation HORAIRE des trajets
- ✅ Features simples et explicables (pas de lag, pas de rolling averages)
- ✅ Train sur 2024 → Test sur 2025

---

## 📋 Table des Matières

1. [Data Loading](#1-data-loading)
2. [Hourly Aggregation](#2-hourly-aggregation)
3. [Temporal Features](#3-temporal-features)
4. [Weather Features](#4-weather-features)
5. [Calendar Features](#5-calendar-features)
6. [Final Dataset](#6-final-dataset)
7. [Train/Test Split](#7-train-test-split)

---

## 1. Data Loading 📊

Charger les données préparées dans l'EDA

In [1]:
# Import des bibliothèques
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings
from pathlib import Path
from datetime import datetime

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

print("✅ Bibliothèques chargées!")

✅ Bibliothèques chargées!


In [2]:
# Chemins
DATA_PATH = Path('../data')
RAW_PATH = DATA_PATH / 'raw'
PROCESSED_PATH = DATA_PATH / 'processed'

DIVVY_2024_PATH = RAW_PATH / 'divvy' / '2024'
DIVVY_2025_PATH = RAW_PATH / 'divvy' / '2025'
WEATHER_PATH = RAW_PATH / 'weather'
HOLIDAYS_PATH = RAW_PATH / 'holidays'

# Créer le dossier processed si nécessaire
PROCESSED_PATH.mkdir(exist_ok=True)

print(f"📁 Paths configurés")
print(f"   2024 data: {DIVVY_2024_PATH}")
print(f"   2025 data: {DIVVY_2025_PATH}")
print(f"   Output: {PROCESSED_PATH}")

📁 Paths configurés
   2024 data: ..\data\raw\divvy\2024
   2025 data: ..\data\raw\divvy\2025
   Output: ..\data\processed


### 1.1 Charger Divvy 2024 (Train)

In [3]:
# Charger tous les fichiers 2024
print("📂 Chargement Divvy 2024...")
divvy_files_2024 = sorted(DIVVY_2024_PATH.glob('*.csv'))
print(f"   Fichiers trouvés: {len(divvy_files_2024)}")

dfs_2024 = []
for file in divvy_files_2024:
    df_month = pd.read_csv(file)
    dfs_2024.append(df_month)
    print(f"   ✓ {file.name}: {len(df_month):,} rows")

df_divvy_2024 = pd.concat(dfs_2024, ignore_index=True)
print(f"\n✅ Total 2024: {len(df_divvy_2024):,} trajets")

📂 Chargement Divvy 2024...
   Fichiers trouvés: 12
   ✓ 202401-divvy-tripdata.csv: 144,873 rows
   ✓ 202402-divvy-tripdata.csv: 223,164 rows
   ✓ 202403-divvy-tripdata.csv: 301,687 rows
   ✓ 202404-divvy-tripdata.csv: 415,025 rows
   ✓ 202405-divvy-tripdata.csv: 609,493 rows
   ✓ 202406-divvy-tripdata.csv: 710,721 rows
   ✓ 202407-divvy-tripdata.csv: 748,962 rows
   ✓ 202408-divvy-tripdata.csv: 755,639 rows
   ✓ 202409-divvy-tripdata.csv: 821,276 rows
   ✓ 202410-divvy-tripdata.csv: 616,281 rows
   ✓ 202411-divvy-tripdata.csv: 335,075 rows
   ✓ 202412-divvy-tripdata.csv: 178,372 rows

✅ Total 2024: 5,860,568 trajets


### 1.2 Charger Divvy 2025 (Test)

In [4]:
# Charger les fichiers 2025 disponibles
print("📂 Chargement Divvy 2025...")
divvy_files_2025 = sorted(DIVVY_2025_PATH.glob('*.csv'))
print(f"   Fichiers trouvés: {len(divvy_files_2025)}")

if len(divvy_files_2025) > 0:
    dfs_2025 = []
    for file in divvy_files_2025:
        df_month = pd.read_csv(file)
        dfs_2025.append(df_month)
        print(f"   ✓ {file.name}: {len(df_month):,} rows")
    
    df_divvy_2025 = pd.concat(dfs_2025, ignore_index=True)
    print(f"\n✅ Total 2025: {len(df_divvy_2025):,} trajets")
else:
    print("   ⚠️  Aucun fichier 2025 trouvé (sera ajouté plus tard)")
    df_divvy_2025 = None

📂 Chargement Divvy 2025...
   Fichiers trouvés: 12
   ✓ 202501-divvy-tripdata.csv: 138,689 rows
   ✓ 202502-divvy-tripdata.csv: 151,880 rows
   ✓ 202503-divvy-tripdata.csv: 298,155 rows
   ✓ 202504-divvy-tripdata.csv: 371,341 rows
   ✓ 202505-divvy-tripdata.csv: 502,456 rows
   ✓ 202506-divvy-tripdata.csv: 678,904 rows
   ✓ 202507-divvy-tripdata.csv: 763,432 rows
   ✓ 202508-divvy-tripdata.csv: 790,177 rows
   ✓ 202509-divvy-tripdata.csv: 714,759 rows
   ✓ 202510-divvy-tripdata.csv: 646,039 rows
   ✓ 202511-divvy-tripdata.csv: 356,628 rows
   ✓ 202512-divvy-tripdata.csv: 140,534 rows

✅ Total 2025: 5,552,994 trajets


### 1.3 Charger données météo

In [5]:
# Charger météo 2024 et 2025
df_weather_2024 = pd.read_csv(WEATHER_PATH / '2024_weather_chicago.csv')
df_weather_2025 = pd.read_csv(WEATHER_PATH / '2025_weather_chicago.csv')

# Combiner
df_weather = pd.concat([df_weather_2024, df_weather_2025], ignore_index=True)

# Préparer la date
df_weather['date'] = pd.to_datetime(df_weather['date']).dt.date

print(f"✅ Weather data loaded: {len(df_weather)} jours")
print(f"   Colonnes: {df_weather.columns.tolist()}")
df_weather.head()

✅ Weather data loaded: 731 jours
   Colonnes: ['date', 'tavg', 'tmin', 'tmax', 'prcp', 'snow', 'wdir', 'wspd', 'wpgt', 'pres', 'tsun']


,date,tavg,tmin,tmax,prcp,snow,wdir,wspd,wpgt,pres,tsun
0,2024-01-01,-1.0,-2.8,0.0,0.0,NaN,NaN,15.8,NaN,1025.8,NaN
1,2024-01-02,-0.5,-3.3,2.8,0.0,NaN,NaN,19.3,NaN,1021.6,NaN
2,2024-01-03,0.9,-0.6,2.2,0.0,NaN,NaN,11.0,NaN,1019.3,NaN
3,2024-01-04,-0.5,-3.3,1.7,0.0,NaN,NaN,7.1,NaN,1027.3,NaN
4,2024-01-05,-0.4,-4.4,2.8,0.0,NaN,NaN,7.6,NaN,1023.0,NaN


### 1.4 Charger jours fériés

In [6]:
# Charger jours fériés
df_holidays = pd.read_csv(HOLIDAYS_PATH / 'us_holidays_2024_2025.csv')
df_holidays['date'] = pd.to_datetime(df_holidays['date']).dt.date
holiday_dates = df_holidays['date'].tolist()

print(f"✅ Holidays loaded: {len(df_holidays)} jours fériés")
print(f"\n📅 Aperçu:")
df_holidays.head(10)

✅ Holidays loaded: 22 jours fériés

📅 Aperçu:


,date,holiday_name,type
0,2024-01-01,New Year's Day,federal
1,2024-01-15,Martin Luther King Jr. Day,federal
2,2024-02-19,Washington's Birthday,federal
3,2024-05-27,Memorial Day,federal
4,2024-06-19,Juneteenth National Independence Day,federal
5,2024-07-04,Independence Day,federal
6,2024-09-02,Labor Day,federal
7,2024-10-14,Columbus Day,federal
8,2024-11-11,Veterans Day,federal
9,2024-11-28,Thanksgiving Day,federal


---

## 2. Hourly Aggregation ⏰

**Objectif**: Créer un dataset avec le nombre de trajets PAR HEURE

### 2.1 Préparer les données 2024

In [7]:
# Convertir les dates (format ISO8601 pour gérer les millisecondes)
print("⏳ Conversion des dates 2024...")
df_divvy_2024['started_at'] = pd.to_datetime(df_divvy_2024['started_at'], format='ISO8601')

# Créer une colonne datetime_hour (arrondie à l'heure)
df_divvy_2024['datetime_hour'] = df_divvy_2024['started_at'].dt.floor('H')

print("✅ Dates converties")
print(f"   Période: {df_divvy_2024['started_at'].min()} → {df_divvy_2024['started_at'].max()}")

⏳ Conversion des dates 2024...
✅ Dates converties
   Période: 2024-01-01 00:00:39 → 2024-12-31 23:56:49.854000


In [8]:
# Agrégation HORAIRE
print("🔄 Agrégation horaire 2024...")
df_hourly_2024 = df_divvy_2024.groupby('datetime_hour').agg({
    'ride_id': 'count'  # Nombre de trajets
}).reset_index()

df_hourly_2024.columns = ['datetime_hour', 'trip_count']

print(f"✅ Agrégation terminée: {len(df_hourly_2024):,} heures")
print(f"\n📊 Statistiques:")
print(df_hourly_2024['trip_count'].describe())
df_hourly_2024.head()

🔄 Agrégation horaire 2024...
✅ Agrégation terminée: 8,782 heures

📊 Statistiques:
count    8782.000000
mean      667.338647
std       667.821275
min         1.000000
25%       131.000000
50%       442.500000
75%       997.000000
max      3789.000000
Name: trip_count, dtype: float64


,datetime_hour,trip_count
0,2024-01-01 00:00:00,180
1,2024-01-01 01:00:00,373
2,2024-01-01 02:00:00,238
3,2024-01-01 03:00:00,49
4,2024-01-01 04:00:00,23


### 2.2 Préparer les données 2025 (si disponibles)

In [9]:
# Préparer 2025 si disponible
if df_divvy_2025 is not None:
    print("⏳ Conversion des dates 2025...")
    df_divvy_2025['started_at'] = pd.to_datetime(df_divvy_2025['started_at'], format='ISO8601')
    df_divvy_2025['datetime_hour'] = df_divvy_2025['started_at'].dt.floor('H')
    
    # Filtrer pour ne garder QUE les dates 2025
    print("🔍 Filtrage des données 2025...")
    avant_filtre = len(df_divvy_2025)
    df_divvy_2025 = df_divvy_2025[df_divvy_2025['started_at'].dt.year == 2025]
    apres_filtre = len(df_divvy_2025)
    print(f"   Lignes avant filtre: {avant_filtre:,}")
    print(f"   Lignes après filtre: {apres_filtre:,}")
    print(f"   Lignes supprimées: {avant_filtre - apres_filtre:,}")
    
    print("🔄 Agrégation horaire 2025...")
    df_hourly_2025 = df_divvy_2025.groupby('datetime_hour').agg({
        'ride_id': 'count'
    }).reset_index()
    df_hourly_2025.columns = ['datetime_hour', 'trip_count']
    
    print(f"✅ Agrégation 2025 terminée: {len(df_hourly_2025):,} heures")
    print(f"   Période: {df_hourly_2025['datetime_hour'].min()} → {df_hourly_2025['datetime_hour'].max()}")
    display(df_hourly_2025.head())
else:
    print("⚠️  Pas de données 2025 pour l'instant")
    df_hourly_2025 = None

⏳ Conversion des dates 2025...
🔍 Filtrage des données 2025...
   Lignes avant filtre: 5,552,994
   Lignes après filtre: 5,552,941
   Lignes supprimées: 53
🔄 Agrégation horaire 2025...
✅ Agrégation 2025 terminée: 8,758 heures
   Période: 2025-01-01 00:00:00 → 2025-12-31 23:00:00


,datetime_hour,trip_count
0,2025-01-01 00:00:00,336
1,2025-01-01 01:00:00,436
2,2025-01-01 02:00:00,213
3,2025-01-01 03:00:00,57
4,2025-01-01 04:00:00,24


---

## 3. Temporal Features 📅

Créer des features temporelles simples et explicables

In [10]:
def create_temporal_features(df):
    """
    Créer des features temporelles à partir de datetime_hour
    
    Features créées:
    - hour: heure de la journée (0-23)
    - day_of_week: jour de la semaine (0=lundi, 6=dimanche)
    - month: mois (1-12)
    - is_weekend: 1 si weekend, 0 sinon
    - season: saison (winter, spring, summer, fall)
    - date: date (pour merge avec météo)
    """
    df = df.copy()
    
    # Features basiques
    df['hour'] = df['datetime_hour'].dt.hour
    df['day_of_week'] = df['datetime_hour'].dt.dayofweek
    df['month'] = df['datetime_hour'].dt.month
    df['date'] = df['datetime_hour'].dt.date
    
    # Weekend (samedi=5, dimanche=6)
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
    
    # Saison (simple mapping par mois)
    season_map = {
        12: 'winter', 1: 'winter', 2: 'winter',
        3: 'spring', 4: 'spring', 5: 'spring',
        6: 'summer', 7: 'summer', 8: 'summer',
        9: 'fall', 10: 'fall', 11: 'fall'
    }
    df['season'] = df['month'].map(season_map)
    
    return df

print("✅ Fonction create_temporal_features définie")

✅ Fonction create_temporal_features définie


In [11]:
# Appliquer sur 2024
print("🔧 Création des features temporelles 2024...")
df_hourly_2024 = create_temporal_features(df_hourly_2024)

print("✅ Features temporelles créées:")
print(f"   Colonnes: {df_hourly_2024.columns.tolist()}")
print(f"\n📊 Aperçu:")
df_hourly_2024.head()

🔧 Création des features temporelles 2024...
✅ Features temporelles créées:
   Colonnes: ['datetime_hour', 'trip_count', 'hour', 'day_of_week', 'month', 'date', 'is_weekend', 'season']

📊 Aperçu:


,datetime_hour,trip_count,hour,day_of_week,month,date,is_weekend,season
0,2024-01-01 00:00:00,180,0,0,1,2024-01-01,0,winter
1,2024-01-01 01:00:00,373,1,0,1,2024-01-01,0,winter
2,2024-01-01 02:00:00,238,2,0,1,2024-01-01,0,winter
3,2024-01-01 03:00:00,49,3,0,1,2024-01-01,0,winter
4,2024-01-01 04:00:00,23,4,0,1,2024-01-01,0,winter


In [12]:
# Appliquer sur 2025 si disponible
if df_hourly_2025 is not None:
    print("🔧 Création des features temporelles 2025...")
    df_hourly_2025 = create_temporal_features(df_hourly_2025)
    print("✅ Features temporelles 2025 créées")
    print(f"\n📊 Aperçu:")
    display(df_hourly_2025.head())

🔧 Création des features temporelles 2025...
✅ Features temporelles 2025 créées

📊 Aperçu:


,datetime_hour,trip_count,hour,day_of_week,month,date,is_weekend,season
0,2025-01-01 00:00:00,336,0,2,1,2025-01-01,0,winter
1,2025-01-01 01:00:00,436,1,2,1,2025-01-01,0,winter
2,2025-01-01 02:00:00,213,2,2,1,2025-01-01,0,winter
3,2025-01-01 03:00:00,57,3,2,1,2025-01-01,0,winter
4,2025-01-01 04:00:00,24,4,2,1,2025-01-01,0,winter


### 3.1 Vérification des features temporelles

In [13]:
# Vérifier la distribution des features
print("📊 Distribution des features temporelles (2024):")
print(f"\nHour: {df_hourly_2024['hour'].min()} → {df_hourly_2024['hour'].max()}")
print(f"Day of week: {df_hourly_2024['day_of_week'].min()} → {df_hourly_2024['day_of_week'].max()}")
print(f"Month: {df_hourly_2024['month'].min()} → {df_hourly_2024['month'].max()}")
print(f"\nSeasons: {df_hourly_2024['season'].value_counts().to_dict()}")
print(f"Weekend ratio: {df_hourly_2024['is_weekend'].mean():.2%}")

📊 Distribution des features temporelles (2024):

Hour: 0 → 23
Day of week: 0 → 6
Month: 1 → 12

Seasons: {'summer': 2208, 'spring': 2207, 'fall': 2184, 'winter': 2183}
Weekend ratio: 28.41%


---

## 4. Weather Features 🌤️

Merger les données météo (quotidiennes) avec les données horaires

### 4.1 Sélectionner les features météo importantes

In [14]:
# Vérifier les colonnes météo disponibles
print("🌤️  Colonnes météo disponibles:")
print(df_weather.columns.tolist())
print(f"\n📊 Statistiques météo:")
df_weather.describe()

🌤️  Colonnes météo disponibles:
['date', 'tavg', 'tmin', 'tmax', 'prcp', 'snow', 'wdir', 'wspd', 'wpgt', 'pres', 'tsun']

📊 Statistiques météo:


,tavg,tmin,tmax,prcp,snow,wdir,wspd,wpgt,pres,tsun
count,731.000000,731.000000,731.000000,526.00000,0.0,0.0,731.000000,1.0,730.000000,0.0
mean,11.595212,6.893434,16.175923,0.90057,NaN,NaN,12.706703,28.0,1016.655890,NaN
std,10.697211,10.388528,11.512190,3.21903,NaN,NaN,5.528717,NaN,7.049609,NaN
min,-21.000000,-23.300000,-16.700000,0.00000,NaN,NaN,2.300000,28.0,988.300000,NaN
25%,3.700000,-0.600000,7.200000,0.00000,NaN,NaN,8.400000,28.0,1012.000000,NaN
50%,12.300000,7.800000,18.000000,0.00000,NaN,NaN,12.000000,28.0,1016.950000,NaN
75%,20.900000,15.000000,26.100000,0.00000,NaN,NaN,16.250000,28.0,1021.200000,NaN
max,31.200000,27.200000,35.600000,24.00000,NaN,NaN,34.300000,28.0,1038.800000,NaN


In [15]:
# Sélectionner et renommer les colonnes importantes
# Adapter selon les colonnes réelles dans le CSV météo
weather_features = df_weather[['date']].copy()

# Ajouter les features météo (adapter les noms de colonnes)
# Exemple: si les colonnes sont 'tavg', 'prcp', 'wspd'
if 'tavg' in df_weather.columns:
    weather_features['temperature'] = df_weather['tavg']
if 'prcp' in df_weather.columns:
    weather_features['precipitation'] = df_weather['prcp']
if 'wspd' in df_weather.columns:
    weather_features['wind_speed'] = df_weather['wspd']

print("✅ Weather features sélectionnées:")
print(weather_features.columns.tolist())
weather_features.head()

✅ Weather features sélectionnées:
['date', 'temperature', 'precipitation', 'wind_speed']


,date,temperature,precipitation,wind_speed
0,2024-01-01,-1.0,0.0,15.8
1,2024-01-02,-0.5,0.0,19.3
2,2024-01-03,0.9,0.0,11.0
3,2024-01-04,-0.5,0.0,7.1
4,2024-01-05,-0.4,0.0,7.6


### 4.2 Merger avec les données horaires

In [16]:
# Merger 2024
print("🔗 Merge météo avec données horaires 2024...")
df_hourly_2024 = df_hourly_2024.merge(weather_features, on='date', how='left')

print(f"✅ Merge terminé: {len(df_hourly_2024)} rows")
print(f"\n⚠️  Missing values après merge:")
print(df_hourly_2024[['temperature', 'precipitation', 'wind_speed']].isnull().sum())
df_hourly_2024.head()

🔗 Merge météo avec données horaires 2024...


✅ Merge terminé: 8782 rows

⚠️  Missing values après merge:
temperature         0
precipitation    3648
wind_speed          0
dtype: int64


,datetime_hour,trip_count,hour,day_of_week,month,date,is_weekend,season,temperature,precipitation,wind_speed
0,2024-01-01 00:00:00,180,0,0,1,2024-01-01,0,winter,-1.0,0.0,15.8
1,2024-01-01 01:00:00,373,1,0,1,2024-01-01,0,winter,-1.0,0.0,15.8
2,2024-01-01 02:00:00,238,2,0,1,2024-01-01,0,winter,-1.0,0.0,15.8
3,2024-01-01 03:00:00,49,3,0,1,2024-01-01,0,winter,-1.0,0.0,15.8
4,2024-01-01 04:00:00,23,4,0,1,2024-01-01,0,winter,-1.0,0.0,15.8


In [17]:
# Merger 2025 si disponible
if df_hourly_2025 is not None:
    print("🔗 Merge météo avec données horaires 2025...")
    df_hourly_2025 = df_hourly_2025.merge(weather_features, on='date', how='left')
    print(f"✅ Merge 2025 terminé: {len(df_hourly_2025)} rows")
    print(f"\n⚠️  Missing values après merge:")
    #print(df_hourly_2025[['temperature', 'precipitation', 'wind_speed']].isnull().sum())
display(df_hourly_2025.head())

🔗 Merge météo avec données horaires 2025...
✅ Merge 2025 terminé: 8758 rows

⚠️  Missing values après merge:


,datetime_hour,trip_count,hour,day_of_week,month,date,is_weekend,season,temperature,precipitation,wind_speed
0,2025-01-01 00:00:00,336,0,2,1,2025-01-01,0,winter,-1.5,NaN,21.0
1,2025-01-01 01:00:00,436,1,2,1,2025-01-01,0,winter,-1.5,NaN,21.0
2,2025-01-01 02:00:00,213,2,2,1,2025-01-01,0,winter,-1.5,NaN,21.0
3,2025-01-01 03:00:00,57,3,2,1,2025-01-01,0,winter,-1.5,NaN,21.0
4,2025-01-01 04:00:00,24,4,2,1,2025-01-01,0,winter,-1.5,NaN,21.0


### 4.3 Gérer les valeurs manquantes météo

In [18]:
# Remplacer les valeurs manquantes par 0 (logique: pas de données = pas de pluie/vent/etc)
weather_cols = ['temperature', 'precipitation', 'wind_speed']

for col in weather_cols:
    if col in df_hourly_2024.columns:
        missing_count = df_hourly_2024[col].isnull().sum()
        if missing_count > 0:
            df_hourly_2024[col].fillna(0, inplace=True)
            print(f"✓ {col}: {missing_count} valeurs manquantes remplacées par 0")

# Gérer 2025 aussi
if df_hourly_2025 is not None:
    for col in weather_cols:
        if col in df_hourly_2025.columns:
            missing_count_2025 = df_hourly_2025[col].isnull().sum()
            if missing_count_2025 > 0:
                df_hourly_2025[col].fillna(0, inplace=True)
                print(f"✓ {col} (2025): {missing_count_2025} valeurs manquantes remplacées par 0")

print("\n✅ Toutes les valeurs manquantes météo gérées (remplacées par 0)")

✓ precipitation: 3648 valeurs manquantes remplacées par 0
✓ precipitation (2025): 1271 valeurs manquantes remplacées par 0

✅ Toutes les valeurs manquantes météo gérées (remplacées par 0)


---

## 5. Calendar Features 📅

Ajouter l'indicateur de jours fériés

In [19]:
# Ajouter is_holiday
print("📅 Ajout de l'indicateur de jours fériés...")
df_hourly_2024['is_holiday'] = df_hourly_2024['date'].isin(holiday_dates).astype(int)

holiday_count = df_hourly_2024['is_holiday'].sum()
print(f"✅ is_holiday ajouté: {holiday_count} heures marquées comme férié")
print(f"   Ratio: {df_hourly_2024['is_holiday'].mean():.2%}")

📅 Ajout de l'indicateur de jours fériés...
✅ is_holiday ajouté: 264 heures marquées comme férié
   Ratio: 3.01%


In [20]:
# Ajouter pour 2025 si disponible
if df_hourly_2025 is not None:
    df_hourly_2025['is_holiday'] = df_hourly_2025['date'].isin(holiday_dates).astype(int)
    print(f"✅ is_holiday ajouté pour 2025")

✅ is_holiday ajouté pour 2025


---

## 6. Final Dataset 🎯

Préparer le dataset final pour la modélisation

### 6.1 Sélectionner les colonnes finales

In [21]:
# Colonnes à garder pour la modélisation
feature_cols = [
    'datetime_hour',  # Pour référence temporelle
    'trip_count',     # TARGET
    # Temporal
    'hour',
    'day_of_week',
    'month',
    'is_weekend',
    'season',
    # Weather
    'temperature',
    'precipitation',
    'wind_speed',
    # Calendar
    'is_holiday'
]

df_final_2024 = df_hourly_2024[feature_cols].copy()
print(f"✅ Dataset final 2024: {df_final_2024.shape}")
print(f"\nColonnes: {df_final_2024.columns.tolist()}")
df_final_2024.head()

✅ Dataset final 2024: (8782, 11)

Colonnes: ['datetime_hour', 'trip_count', 'hour', 'day_of_week', 'month', 'is_weekend', 'season', 'temperature', 'precipitation', 'wind_speed', 'is_holiday']


,datetime_hour,trip_count,hour,day_of_week,month,is_weekend,season,temperature,precipitation,wind_speed,is_holiday
0,2024-01-01 00:00:00,180,0,0,1,0,winter,-1.0,0.0,15.8,1
1,2024-01-01 01:00:00,373,1,0,1,0,winter,-1.0,0.0,15.8,1
2,2024-01-01 02:00:00,238,2,0,1,0,winter,-1.0,0.0,15.8,1
3,2024-01-01 03:00:00,49,3,0,1,0,winter,-1.0,0.0,15.8,1
4,2024-01-01 04:00:00,23,4,0,1,0,winter,-1.0,0.0,15.8,1


In [22]:
# Préparer 2025 si disponible
if df_hourly_2025 is not None:
    df_final_2025 = df_hourly_2025[feature_cols].copy()
    print(f"✅ Dataset final 2025: {df_final_2025.shape}")
    display(df_final_2025.head())
else:
    df_final_2025 = None
    print("⚠️  Pas de dataset 2025 pour l'instant")

✅ Dataset final 2025: (8758, 11)


,datetime_hour,trip_count,hour,day_of_week,month,is_weekend,season,temperature,precipitation,wind_speed,is_holiday
0,2025-01-01 00:00:00,336,0,2,1,0,winter,-1.5,0.0,21.0,1
1,2025-01-01 01:00:00,436,1,2,1,0,winter,-1.5,0.0,21.0,1
2,2025-01-01 02:00:00,213,2,2,1,0,winter,-1.5,0.0,21.0,1
3,2025-01-01 03:00:00,57,3,2,1,0,winter,-1.5,0.0,21.0,1
4,2025-01-01 04:00:00,24,4,2,1,0,winter,-1.5,0.0,21.0,1


### 6.2 Vérifications finales

In [23]:
# Vérifier les valeurs manquantes
print("🔍 Vérification des valeurs manquantes:")
missing = df_final_2024.isnull().sum()
if missing.sum() > 0:
    print("⚠️  Valeurs manquantes détectées:")
    print(missing[missing > 0])
else:
    print("✅ Aucune valeur manquante!")

# Statistiques descriptives
print("\n📊 Statistiques descriptives:")
df_final_2024.describe()

🔍 Vérification des valeurs manquantes:
✅ Aucune valeur manquante!

📊 Statistiques descriptives:


,datetime_hour,trip_count,hour,day_of_week,month,is_weekend,temperature,precipitation,wind_speed,is_holiday
count,8782,8782.000000,8782.000000,8782.000000,8782.000000,8782.000000,8782.000000,8782.000000,8782.000000,8782.000000
mean,2024-07-02 00:16:02.924162816,667.338647,11.502277,2.986108,6.514689,0.284104,12.301412,0.261535,12.652961,0.030061
min,2024-01-01 00:00:00,1.000000,0.000000,0.000000,1.000000,0.000000,-21.000000,0.000000,2.700000,0.000000
25%,2024-04-01 13:15:00,131.000000,6.000000,1.000000,4.000000,0.000000,3.800000,0.000000,8.300000,0.000000
50%,2024-07-02 00:30:00,442.500000,12.000000,3.000000,7.000000,0.000000,14.100000,0.000000,12.000000,0.000000
75%,2024-10-01 11:45:00,997.000000,17.750000,5.000000,10.000000,1.000000,21.000000,0.000000,16.200000,0.000000
max,2024-12-31 23:00:00,3789.000000,23.000000,6.000000,12.000000,1.000000,30.400000,24.000000,29.700000,1.000000
std,NaN,667.821275,6.921719,2.003422,3.451118,0.451012,10.077546,1.848902,5.486283,0.170766


In [24]:
# Vérifier la distribution de la target
print("🎯 Distribution de trip_count (TARGET):")
print(df_final_2024['trip_count'].describe())

# Visualisation
fig = px.histogram(df_final_2024, x='trip_count', nbins=50,
                   title='Distribution du nombre de trajets par heure',
                   labels={'trip_count': 'Nombre de trajets'})
fig.show()

🎯 Distribution de trip_count (TARGET):
count    8782.000000
mean      667.338647
std       667.821275
min         1.000000
25%       131.000000
50%       442.500000
75%       997.000000
max      3789.000000
Name: trip_count, dtype: float64


### 6.3 Encoder les variables catégorielles

In [25]:
# Encoder 'season' en one-hot
print("🔧 Encodage de la variable 'season'...")
season_dummies_2024 = pd.get_dummies(df_final_2024['season'], prefix='season', drop_first=False)
df_final_2024 = pd.concat([df_final_2024.drop('season', axis=1), season_dummies_2024], axis=1)

print(f"✅ Encoding terminé")
print(f"   Nouvelles colonnes: {season_dummies_2024.columns.tolist()}")
print(f"\n📊 Shape finale: {df_final_2024.shape}")
df_final_2024.head()

🔧 Encodage de la variable 'season'...
✅ Encoding terminé
   Nouvelles colonnes: ['season_fall', 'season_spring', 'season_summer', 'season_winter']

📊 Shape finale: (8782, 14)


,datetime_hour,trip_count,hour,day_of_week,month,is_weekend,temperature,precipitation,wind_speed,is_holiday,season_fall,season_spring,season_summer,season_winter
0,2024-01-01 00:00:00,180,0,0,1,0,-1.0,0.0,15.8,1,False,False,False,True
1,2024-01-01 01:00:00,373,1,0,1,0,-1.0,0.0,15.8,1,False,False,False,True
2,2024-01-01 02:00:00,238,2,0,1,0,-1.0,0.0,15.8,1,False,False,False,True
3,2024-01-01 03:00:00,49,3,0,1,0,-1.0,0.0,15.8,1,False,False,False,True
4,2024-01-01 04:00:00,23,4,0,1,0,-1.0,0.0,15.8,1,False,False,False,True


In [26]:
# Encoder 2025 si disponible
if df_final_2025 is not None:
    season_dummies_2025 = pd.get_dummies(df_final_2025['season'], prefix='season', drop_first=False)
    df_final_2025 = pd.concat([df_final_2025.drop('season', axis=1), season_dummies_2025], axis=1)
    print(f"✅ Encoding 2025 terminé: {df_final_2025.shape}")

✅ Encoding 2025 terminé: (8758, 14)


---

## 7. Train/Test Split 🔀

Sauvegarder les datasets train (2024) et test (2025)

### 7.1 Sauvegarder les datasets

In [27]:
# Sauvegarder 2024 (train)
output_file_2024 = PROCESSED_PATH / 'train_2024_hourly.csv'
df_final_2024.to_csv(output_file_2024, index=False)
print(f"✅ Dataset TRAIN sauvegardé: {output_file_2024}")
print(f"   Shape: {df_final_2024.shape}")
print(f"   Size: {output_file_2024.stat().st_size / 1024**2:.2f} MB")

✅ Dataset TRAIN sauvegardé: ..\data\processed\train_2024_hourly.csv
   Shape: (8782, 14)
   Size: 0.61 MB


In [28]:
# Sauvegarder 2025 (test) si disponible
if df_final_2025 is not None:
    output_file_2025 = PROCESSED_PATH / 'test_2025_hourly.csv'
    df_final_2025.to_csv(output_file_2025, index=False)
    print(f"✅ Dataset TEST sauvegardé: {output_file_2025}")
    print(f"   Shape: {df_final_2025.shape}")
    print(f"   Size: {output_file_2025.stat().st_size / 1024**2:.2f} MB")
else:
    print("ℹ️  Pas de dataset 2025 à sauvegarder pour l'instant")

✅ Dataset TEST sauvegardé: ..\data\processed\test_2025_hourly.csv
   Shape: (8758, 14)
   Size: 0.60 MB


### 7.2 Résumé final

In [29]:
# Résumé complet
print("="*80)
print("📋 RÉSUMÉ - FEATURE ENGINEERING")
print("="*80)

print(f"\n✅ Dataset TRAIN (2024):")
print(f"   Shape: {df_final_2024.shape}")
print(f"   Période: {df_final_2024['datetime_hour'].min()} → {df_final_2024['datetime_hour'].max()}")
print(f"   Target (trip_count): min={df_final_2024['trip_count'].min()}, "
      f"max={df_final_2024['trip_count'].max()}, "
      f"mean={df_final_2024['trip_count'].mean():.0f}")

if df_final_2025 is not None:
    print(f"\n✅ Dataset TEST (2025):")
    print(f"   Shape: {df_final_2025.shape}")
    print(f"   Période: {df_final_2025['datetime_hour'].min()} → {df_final_2025['datetime_hour'].max()}")

print(f"\n🔧 Features créées ({df_final_2024.shape[1]-2} features + 1 target + 1 datetime):")
feature_list = [col for col in df_final_2024.columns if col not in ['datetime_hour', 'trip_count']]
print(f"   {feature_list}")

print("\n📊 Types de features:")
print(f"   Temporelles: hour, day_of_week, month, is_weekend, season_*")
print(f"   Météo: temperature, precipitation, wind_speed")
print(f"   Calendrier: is_holiday")

print("\n" + "="*80)
print("✅ FEATURE ENGINEERING TERMINÉ!")
print("="*80)
print("\n⏭️  Prochaine étape: Modeling (notebook 03)")

📋 RÉSUMÉ - FEATURE ENGINEERING

✅ Dataset TRAIN (2024):
   Shape: (8782, 14)
   Période: 2024-01-01 00:00:00 → 2024-12-31 23:00:00
   Target (trip_count): min=1, max=3789, mean=667

✅ Dataset TEST (2025):
   Shape: (8758, 14)
   Période: 2025-01-01 00:00:00 → 2025-12-31 23:00:00

🔧 Features créées (12 features + 1 target + 1 datetime):
   ['hour', 'day_of_week', 'month', 'is_weekend', 'temperature', 'precipitation', 'wind_speed', 'is_holiday', 'season_fall', 'season_spring', 'season_summer', 'season_winter']

📊 Types de features:
   Temporelles: hour, day_of_week, month, is_weekend, season_*
   Météo: temperature, precipitation, wind_speed
   Calendrier: is_holiday

✅ FEATURE ENGINEERING TERMINÉ!

⏭️  Prochaine étape: Modeling (notebook 03)


---

## 🎉 Feature Engineering Terminé!

**Date:** 2026-01-13  
**Status:** ✅ Completed  
**Next:** Modeling - Linear Regression, Random Forest, XGBoost

### 📝 Ce qui a été fait:
- ✅ Agrégation horaire des trajets (2024 + 2025)
- ✅ Features temporelles simples et explicables
- ✅ Merge avec données météo quotidiennes
- ✅ Ajout indicateur jours fériés
- ✅ Encoding des variables catégorielles
- ✅ Datasets train/test sauvegardés

### 🎯 Datasets prêts pour la modélisation:
- `train_2024_hourly.csv` : ~8,760 heures (365 jours × 24h)
- `test_2025_hourly.csv` : À venir selon données disponibles

---